## Cell 1: Google Colab Setup & Google Drive Mount

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully")
else:
    print("Running locally (not in Google Colab)")

if IN_COLAB:
    GDRIVE_PROJECT_PATH = '/content/drive/MyDrive/audio_simon_moutier'
else:
    GDRIVE_PROJECT_PATH = 'C:/Users/Simon/Desktop/CNRS'

print(f"Project path: {GDRIVE_PROJECT_PATH}")

## Cell 2: Install Dependencies

In [ ]:
if IN_COLAB:
    print("Installing required packages...")
    !pip install -q pyarrow fastparquet scikit-learn joblib
    print("All packages installed")

## Cell 3: Imports

In [ ]:
import os
import re
import gc
import time
import joblib
import numpy as np
import pandas as pd

from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    f1_score
)

print("Imports OK")

## Cell 4: Configuration & Paths

In [ ]:
if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/audio_simon_moutier'
else:
    PROJECT_PATH = 'C:/Users/Simon/Desktop/CNRS'

PATH_EMBEDDINGS = os.path.join(PROJECT_PATH, 'embeddings/vgg16_embeddings_subset.parquet')
PATH_EXPORT = os.path.join(PROJECT_PATH, 'Results/Random_Forest')
GROUP_LABELS_PATH = os.path.join(PATH_EXPORT, 'grouped_labels_V2.csv')

print(f"Embeddings : {PATH_EMBEDDINGS}")
print(f"Export : {PATH_EXPORT}")

## Cell 5: Options & Parameters

In [ ]:
use_label_group_modified = True
group_noise = True
downsampling = False
use_custom_weights = True
run_tuning = False
ungrouped_labels = False

use_pca_reduction = False
n_pca_components = 0.95
reduce_cv_splits = False


print('Configuration:')
print(f'  - Use label group modified: {use_label_group_modified}')
print(f'  - Group noise: {group_noise}')
print(f'  - Downsampling: {downsampling}')
print(f'  - Custom weights: {use_custom_weights}')
print(f'  - PCA reduction: {use_pca_reduction} ({n_pca_components} components)')
print(f'  - Reduce CV splits: {reduce_cv_splits}')

## Cell 6: Create Output Folder

In [ ]:
start_time = time.time()

print('\n=====================================================')
print(' RANDOM FOREST TRAINING')
print('=====================================================')
print()

print('[1/11] Creating output folder...')

os.makedirs(PATH_EXPORT, exist_ok=True)

existing_rf = [d for d in os.listdir(PATH_EXPORT) if re.match(r'RF_\d+', d)]
rf_numbers = [int(d.split('_')[1]) for d in existing_rf]
next_rf = 1 if len(rf_numbers) == 0 else max(rf_numbers) + 1

PATH_RF_OUTPUT = os.path.join(PATH_EXPORT, f'RF_{next_rf}')
os.makedirs(PATH_RF_OUTPUT, exist_ok=True)

print(f'Output : {PATH_RF_OUTPUT}\n')

## Cell 7: Load Data

In [ ]:
print('[2/11] Loading embeddings...')

start_load = time.time()
df = pd.read_parquet(PATH_EMBEDDINGS)

feature_cols = [c for c in df.columns if c.startswith('dim_')]
df[feature_cols] = df[feature_cols].astype(np.float32)

load_time = time.time() - start_load

print(f'Loaded dataframe shape: {df.shape}')
print(f'Load time: {load_time:.2f} sec')

## Cell 8: Extract Metadata

In [ ]:
print('[3/11] Extracting metadata...')

df['label'] = df['filename'].str.extract(r'(.*)(?=_HiP)')
df['id'] = df['filename'].str.extract(r'(HiP[^_]+)')
df['specie'] = np.where(
    df['id'].isin(['HiPsh441', 'HiPsh435']),
    'hyaena',
    np.where(
        df['id'].isin(['HiP616', 'HiP320', 'HiP633']),
        'lion',
        'unknown'
    )
)

print('Metadata extraction done.\n')

## Cell 9: Filter Rare Classes

In [ ]:
print('[4/11] Filtering rare classes...')

THRESHOLD = 30
counts = df['label'].value_counts()
valid_labels = counts[counts >= THRESHOLD].index.tolist()

initial_n = len(df)
df = df[df['label'].isin(valid_labels)].copy()
final_n = len(df)

print(f'Samples before filtering: {initial_n}')
print(f'Samples after filtering : {final_n}')
print(f'Remaining labels         : {df["label"].nunique()}\n')

## Cell 10: Group Labels

In [ ]:
print('[6/11] Grouping labels...')

if not ungrouped_labels:
    labels_df = pd.read_csv(GROUP_LABELS_PATH)
    group_map = {
        1: 'background',
        2: 'crunch',
        3: 'roar',
        4: 'whoop',
        5: 'prey_scream',
        6: 'h_noise',
        7: 'l_noise',
        8: 'whoop_o',
        9: 'roar_o'
    }
    labels_df['group_name'] = labels_df['group'].map(group_map).fillna('unclassified')
    if group_noise:
        labels_df['group_name'] = labels_df['group_name'].replace({
            'h_noise': 'noise',
            'l_noise': 'noise'
        })
    df = df.merge(labels_df[['label', 'group_name']], on='label', how='left')
else:
    df['group_name'] = df['label']

print('Grouped labels distribution:\n')
print(df['group_name'].value_counts())
print()

## Cell 11: Prepare Features & PCA

In [ ]:
print('[8/11] Preparing features...')

X = df[feature_cols]
y = df['group_name']

print(f'Number of features (original): {len(feature_cols)}')
print(f'Feature matrix shape: {X.shape}')

if use_pca_reduction:
    print(f'\nApplying PCA reduction to {n_pca_components} components...')
    pca = PCA(n_components=n_pca_components, random_state=RANDOM_STATE)
    X = pca.fit_transform(X).astype(np.float32)
    print(f'Explained variance ratio: {pca.explained_variance_ratio_.sum():.4f}')
    print(f'Features after PCA: {X.shape[1]}')
    pca_path = os.path.join(PATH_RF_OUTPUT, f'RF_{next_rf}_pca.pkl')
    joblib.dump(pca, pca_path)
    print(f'PCA saved : {pca_path}')
    gc.collect()
else:
    X = X.values.astype(np.float32)

print(f'\nFinal feature matrix shape: {X.shape}\n')

## Cell 12: Compute Weights

In [ ]:
print('[9/11] Computing sample weights...')

class_counts = y.value_counts()

if not use_custom_weights:
    weights = y.map(lambda x: 1 / class_counts[x])
else:
    priorite = {
        'background': 1,
        'noise': 1,
        'crunch': 1,
        'roar': 1.5,
        'roar_o': 1.5,
        'whoop': 2,
        'whoop_o': 2
    }
    weights = y.map(lambda x: (1 / class_counts[x]) * priorite.get(x, 1))

weights = weights / weights.mean()
weights = weights.values.astype(np.float32)

print('Weights computed.\n')

## Cell 13: Setup GroupKFold & Random Forest

In [ ]:
print('[10/11] Preparing GroupKFold...')

df['original_file_id'] = df['filename'].str.extract(r'(HiP.+?)(?=_idx)')
groups = df['original_file_id']

cv_splits = 3 if reduce_cv_splits else 5
gkf = GroupKFold(n_splits=cv_splits)

print(f'GroupKFold ready with {cv_splits} splits.\n')

print('[11/11] Initializing Random Forest...')

rf = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    criterion='gini',
    min_samples_leaf=5,
    n_jobs=-1,
    class_weight=None,
    verbose=1,
    random_state=42
)

print('Model initialized (matching train_audio_model.R):')
print(f'  - n_estimators: 200')
print(f'  - max_features: sqrt')
print('  - criterion: gini')
print(f'  - min_samples_leaf: 5\n')

## Cell 14: Cross-Validation Training

In [ ]:
print('\n=====================================================')
print(' CROSS VALIDATION')
print('=====================================================')
print()

y_pred = np.empty(len(y), dtype=object)
fold = 1
cv_start = time.time()

for train_idx, test_idx in gkf.split(X, y, groups):
    fold_start = time.time()
    print(f'Fold {fold}/{cv_splits}')
    print('-' * 40)

    X_train = X[train_idx]
    X_test = X[test_idx]
    y_train = y.iloc[train_idx]
    w_train = weights[train_idx]

    print(f'Train samples: {len(train_idx)}')
    print(f'Test samples : {len(test_idx)}')
    print('Training model...')

    rf.fit(X_train, y_train, sample_weight=w_train)

    print('Predicting...')
    y_pred[test_idx] = rf.predict(X_test)

    fold_f1 = f1_score(y.iloc[test_idx], y_pred[test_idx], average='macro')
    elapsed_fold = time.time() - fold_start

    print(f'Fold Macro F1: {fold_f1:.4f}')
    print(f'Fold duration: {elapsed_fold:.2f} sec\n')
    fold += 1

cv_elapsed = time.time() - cv_start
print('Cross-validation completed.')
print(f'Total CV time: {cv_elapsed/60:.2f} minutes\n')

## Cell 15: Evaluation Metrics

In [ ]:
print('\n=====================================================')
print(' EVALUATION')
print('=====================================================')
print()

macro_f1 = f1_score(y, y_pred, average='macro')
balanced_acc = balanced_accuracy_score(y, y_pred)

print(f'Macro F1 Score     : {macro_f1:.4f}')
print(f'Balanced Accuracy  : {balanced_acc:.4f}\n')

cm = confusion_matrix(y, y_pred)
cm_path = os.path.join(PATH_RF_OUTPUT, f'confusion_matrix_RF_{next_rf}.csv')
pd.DataFrame(cm).to_csv(cm_path, index=False)
print(f'Confusion matrix    : {cm_path}')

## Cell 16: Final Model Training

In [ ]:
print('\nTraining final model...')
rf.fit(X, y, sample_weight=weights)
print('Model trained')

## Cell 17: Save Results

In [ ]:
print('Saving outputs...')

model_path = os.path.join(PATH_RF_OUTPUT, f'RF_{next_rf}_model.pkl')
joblib.dump(rf, model_path)
print(f'Model saved to      : {model_path}')

print(f'Confusion matrix    : {cm_path}')

## Cell 18: Generate Summary Report

In [ ]:
summary_path = os.path.join(PATH_RF_OUTPUT, f'summary_RF_{next_rf}.txt')

with open(summary_path, 'w') as f:
    f.write('=' * 60 + '\n')
    f.write(' RANDOM FOREST SUMMARY\n')
    f.write('=' * 60 + '\n\n')
    f.write(f'Date : {datetime.now()}\n\n')
    f.write(f'Macro F1       : {macro_f1:.4f}\n')
    f.write(f'Balanced Acc   : {balanced_acc:.4f}\n\n')
    f.write(f'Samples        : {len(df)}\n')
    f.write(f'Features       : {X.shape[1]}\n')
    f.write(f'Classes        : {y.nunique()}\n\n')
    if use_pca_reduction:
        f.write(f'PCA variance   : {pca.explained_variance_ratio_.sum():.4f}\n\n')
    f.write(classification_report(y, y_pred))

total_elapsed = time.time() - start_time

print('\n=====================================================')
print(' TRAINING COMPLETED')
print('=====================================================')
print(f'Total runtime: {total_elapsed/60:.2f} minutes')
print(f'Results : {PATH_RF_OUTPUT}')
print('=====================================================')